In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Sample data with separated date columns and missing dates
data = {'year': [2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023, 2023],
        'month': [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
        'day': [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]}

# Create the initial dataframe
df = pd.DataFrame(data)

# Create sales, price, and promotion columns
data['sales'] = np.tile(np.arange(2, 8), 3)[:len(data)]
data['promotion'] = np.tile(np.repeat([1, 0], 7), 2)[:len(data)]
data['price'] = 1.5 * data['promotion'] + 2.5 * (1 - data['promotion'])

# Combine date columns into a single datetime column
data['date'] = pd.to_datetime(data[['year', 'month', 'day']])

# Set the datetime column as the index
data.set_index('date', inplace=True)

# Resample the dataframe with daily frequency and create a 'missing' column
resampled_data = data.resample('D').asfreq()
resampled_data['missing'] = resampled_data['sales'].isna().astype(int)

# Fill missing sales values with 0
resampled_data['sales'].fillna(0, inplace=True)

# Impute missing values using MICE
imputer = IterativeImputer(max_iter=10, random_state=0)
resampled_data['sales_imputed'] = imputer.fit_transform(resampled_data['sales_imputed'])

# Fill missing values using seasonal fill
def seasonal_fill(series, freq='7D'):
    filled_series = series.copy()
    for i, value in enumerate(series):
        if pd.isna(value):
            backward_idx = i - 7
            while backward_idx >= 0 and pd.isna(series[backward_idx]):
                backward_idx -= 7
                
            forward_idx = i + 7
            while forward_idx < len(series) and pd.isna(series[forward_idx]):
                forward_idx += 7
                
            if backward_idx >= 0 and forward_idx < len(series):
                filled_series[i] = (series[backward_idx] + series[forward_idx]) / 2
            elif backward_idx >= 0:
                filled_series[i] = series[backward_idx]
            elif forward_idx < len(series):
                filled_series[i] = series[forward_idx]
    return filled_series

resampled_data['sales_seasonal_fill'] = seasonal_fill(resampled_data['sales'])